In [1]:
print("hello")

hello


In [2]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

def loadPdfFilles(path):
    # allow either relative (to workspace root) or absolute paths
    path_obj = Path(path)
    if not path_obj.is_absolute():
        # assume notebook is running from the research folder
        base = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
        path_obj = base / path_obj

    loader=DirectoryLoader(
        str(path_obj),
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents

c:\Users\vivobook\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# loadPdfFilles with relative path; function will resolve to project root
extractedData = loadPdfFilles("data")

Multiple definitions in dictionary at byte 0x5f1b5 for key /Info
Multiple definitions in dictionary at byte 0x5f1c1 for key /Info
Multiple definitions in dictionary at byte 0x5f1cd for key /Info
Multiple definitions in dictionary at byte 0x4560c for key /Info
Multiple definitions in dictionary at byte 0x45618 for key /Info
Multiple definitions in dictionary at byte 0x45624 for key /Info


In [4]:
extractedData # list of pages 

[Document(metadata={'producer': 'macOS Version 11.3.1 (assemblage 20E241) Quartz PDFContext', 'creator': 'HP Scan', 'creationdate': '2025-02-03T13:44:59+00:00', 'moddate': '2026-02-07T21:25:46+01:00', 'source': 'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject2\\chatBotJuridiquesWithRAG-\\data\\AMTJ-parents.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'macOS Version 11.3.1 (assemblage 20E241) Quartz PDFContext', 'creator': 'HP Scan', 'creationdate': '2025-02-03T13:44:59+00:00', 'moddate': '2026-02-07T21:25:46+01:00', 'source': 'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject2\\chatBotJuridiquesWithRAG-\\data\\AMTJ-parents.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'macOS Version 11.3.1 (assemblage 20E241) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260123110706Z00'00'", 'moddate': "D:20260123110706Z00'00'",

In [5]:
len(extractedData) #nbr of pages

24

turning pdfs into images and extract the text and turn it to json

In [6]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os

# ==========================================
# 1. BULLETPROOF PATHING
# ==========================================
current_path = Path.cwd()

# If running from inside the 'research' folder, go up one level
if current_path.name == "research":
    BASE_DIR = current_path.parent 
else:
    # Failsafe: Hardcoded absolute path
    BASE_DIR = Path(r"C:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject2\chatBotJuridiquesWithRAG-")

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

# ==========================================
# 2. CONFIGURATION
# ==========================================
DPI = 150
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 80 

# sanity checks for external dependencies
if not Path(POPPLER_PATH).exists():
    print(f"⚠️  Poppler path {POPPLER_PATH} does not exist. PDF conversion may fail.")

if not Path(pytesseract.pytesseract.tesseract_cmd).exists():
    print(f"⚠️  Tesseract binary {pytesseract.pytesseract.tesseract_cmd} not found. OCR may fail.")

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    total = len(patches)
    for i, patch in enumerate(patches):
        # Progress indicator so you know it's not frozen
        print(f"      -> Extracting text from patch {i+1}/{total}...", end="\r")
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        texts.append(text.strip())
    print() # Move to the next line after finishing the patches
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"\n--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            print(f"⏳ Converting {pdf_path.name}...")
            pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
            for i, page in enumerate(pages, start=1):
                page_path = doc_out / f"page_{i:03d}.png"
                if not page_path.exists():
                    page.save(page_path, "PNG")
            print(f"✅ {doc_id}: {len(pages)} pages processed.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"\n📂 Processing Document: {doc_folder.name}")

        for img_path in pages_images:
            print(f"  📄 Reading {img_path.name}...")
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)
            print(f"  💾 Saved JSON for {img_path.name}")

    print("\n🎉 All tasks completed successfully!")

process_all_pdfs()

--- Working Directory: c:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject2\chatBotJuridiquesWithRAG- ---

--- STEP 1: CONVERTING 13 PDFs TO IMAGES ---
⏳ Converting AMTJ-parents.pdf...
❌ Error converting AMTJ-parents.pdf: [Errno 2] No such file or directory: 'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject2\\chatBotJuridiquesWithRAG-\\data\\pages\\AMTJ-parents\\page_001.png'
⏳ Converting cir 01-retraite complémentaire 1.pdf...
✅ cir 01-retraite complémentaire 1: 1 pages processed.
⏳ Converting cir 2-Pèlerinage.pdf...
✅ cir 2-Pèlerinage: 1 pages processed.
⏳ Converting CIR 7-transport aérien.pdf...
✅ CIR 7-transport aérien: 1 pages processed.
⏳ Converting CIR 8-convention transport.pdf...
✅ CIR 8-convention transport: 1 pages processed.
⏳ Converting Cir don femmes enceintes26.pdf...
✅ Cir don femmes enceintes26: 1 pages processed.
⏳ Converting com 06-convention g_Omrane.pdf...
✅ com 06-convention g_Omrane: 1 pages processed.
⏳ Converting com 4 AMC.

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject2\\chatBotJuridiquesWithRAG-\\artifacts\\ocr\\cir 01-retraite complémentaire 1_page_001_ocr.json'

Setup the embedding moudle 

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return a Multilingual embedding model optimized for Arabic.
    """
    model_name = "intfloat/multilingual-e5-base"
    
    model_kwargs = {'device': 'cpu'} 
    encode_kwargs = {'normalize_embeddings': True} 
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

embedding = download_embeddings()
print("✅ Embeddings model is ready for Arabic!")

C:\Users\vivobook\AppData\Local\Temp\ipykernel_3752\3733159505.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✅ Embeddings model is ready for Arabic!


In [ ]:
vector = embedding.embed_query("الإقتصاد")
print(len(vector))
print(vector)

768
[0.055196527391672134, 0.05592329800128937, 0.013396338559687138, 0.030474532395601273, 0.01198836974799633, -0.05208773910999298, 0.0007374533452093601, -0.040192220360040665, 0.01440285425633192, -0.0037508143577724695, 0.023447250947356224, 0.023675477132201195, 0.15949103236198425, 0.040033288300037384, -0.02711133286356926, -0.04413944110274315, 0.0441996231675148, -0.051685135811567307, 0.055426765233278275, -0.0008227094658650458, 0.054761551320552826, -0.0019403421320021152, 0.042476292699575424, -0.04440610110759735, 0.0009655682952143252, -0.05327524244785309, 0.026688532903790474, 0.007273482158780098, -0.0005164410686120391, 0.03137904033064842, 0.0005725699593313038, -0.04888027161359787, 0.0017705034697428346, 0.0013964641839265823, 0.039450615644454956, 0.07035262137651443, -0.020239785313606262, -0.02500203624367714, 0.07904477417469025, 0.023940294981002808, 0.03044365532696247, 0.023420758545398712, 0.028036922216415405, -0.035178590565919876, 0.028875131160020828

connect to Pinecone DB

In [7]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found in environment. Please set it in .env or system environment.")

index_name = "juridique-rag-index"

pc = Pinecone(api_key=PINECONE_API_KEY)

print("✅ Setup is ready! Connected to Pinecone.")

ImportError: cannot import name 'Pinecone' from 'pinecone' (unknown location)

In [ ]:
print(" Loading multilingual-e5-base model...")

embedding = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Model loaded successfully!")

In [ ]:
if not pc.list_indexes():
    print(f"⏳ Creating a new index called '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=768,  
        metric="cosine"
    )
    print("✅ Index created successfully!")
else:
    existing_indexes = [idx.name for idx in pc.list_indexes()]
    if index_name in existing_indexes:
        print(f"✅ Index '{index_name}' already exists. We will use it.")
    else:
        print(f"⏳ Creating a new index called '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=768,  
            metric="cosine"
        )
        print("✅ Index created successfully!")

In [ ]:
current_path = Path.cwd()
BASE_DIR = current_path.parent if current_path.name == "research" else current_path
OCR_DIR = BASE_DIR / "artifacts" / "ocr"

if not OCR_DIR.exists():
    print(f"❌ OCR directory {OCR_DIR} not found. Run process_all_pdfs first.")
    json_files = []
else:
    json_files = list(OCR_DIR.glob("*.json"))

print(f"📂 Found {len(json_files)} JSON files. Reading them...")

documents = []

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)
        
        for item in content.get('data', []):
            text = item.get('text', '').strip()
            
            if text and len(text) > 2:
                e5_text = f"passage: {text}"
                
                metadata = {
                    "source": content.get('source_pdf', 'Unknown'),
                    "page": content.get('page_image', 'Unknown')
                }
                
                doc = Document(page_content=e5_text, metadata=metadata)
                documents.append(doc)

print(f" Prepared {len(documents)} chunks (paragraphs) ready for Pinecone.")

In [ ]:
print(" Uploading vectors to Pinecone... ")

docsearch = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

print(" DONE! All your legal documents are now vectorized and stored in Pinecone!")

If you want to add  

In [ ]:
from langchain_core.documents import Document

dswith = Document(
    page_content="passage: سلام",
    metadata={"source": "me", "page": "none"} 
)

print("⏳ Adding manual document to Pinecone...")
print(docsearch.add_documents(documents=[dswith]))
print("✅ Document added successfully!")

retriever

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
query = "اتفاقيات مع أطباء"
retrieved_docs = retriever.get_relevant_documents(query)
retrieved_docs